# Final 15-Day Forecast

This notebook generates the final 15-day demand forecast using the trained DeepAR model.

The forecast covers the period from **August 16 to August 30, 2017** for **1,728 Store × Family series**.

> **Execution Environment:** This notebook is intended to run in **Google Colab** using a compatible PyTorch / PyTorch Forecasting environment. A GPU runtime is recommended for compatibility and performance with the trained DeepAR checkpoint.

### Workflow

1. Load the processed data and trained DeepAR checkpoint.
2. Prepare the future inputs required for the forecast horizon.
3. Reconstruct the DeepAR `TimeSeriesDataSet`.
4. Generate the final 15-day forecast.
5. Validate the forecast output.
6. Save the final predictions for downstream analysis and replenishment.

### Final Forecast Output

The final forecast contains:

- **15 forecast dates:** August 16–30, 2017
- **1,728 Store × Family series**
- **25,920 forecast rows**
- Non-negative forecast values after clipping

The forecast is saved as:

- `predictions/final_deepar_forecast.parquet`
- `predictions/final_deepar_forecast.csv`

## 1. Setup and Data Loading

In [1]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

CUDA available: True
Device: Tesla T4


In [2]:
from google.colab import drive

drive.mount("/content/drive")

%cd /content/drive/MyDrive/DeepAR/DeepAR/notebooks

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/DeepAR/DeepAR/notebooks


In [3]:
import pandas as pd
import numpy as np
from pathlib import Path

In [4]:
project_root = Path("..").resolve()

processed_path = project_root / "data" / "processed"
predictions_path = project_root / "predictions"
models_path = project_root / "models"

print("Project root:", project_root)

Project root: /content/drive/MyDrive/DeepAR/DeepAR


## 2. Final Model

DeepAR was selected as the final forecasting model based on the overall validation results, demand-pattern performance, and intermittent family-level analysis.

The final forecast therefore uses the trained DeepAR model to generate demand forecasts for the next 15 days.

## 3. Load Full Available Training Data

After model selection, the final DeepAR forecast is generated using all available historical training data.

The final cutoff is the last date available in the original training dataset. The model then forecasts the following 15 days.

In [5]:
# Load the full available training data
train_df = pd.read_csv(
    project_root / "data" / "raw" / "train.csv"
)

train_df["date"] = pd.to_datetime(train_df["date"])

print("Training shape:", train_df.shape)
print("First date:", train_df["date"].min())
print("Last date:", train_df["date"].max())
print(
    "Unique series:",
    train_df[["store_nbr", "family"]].drop_duplicates().shape[0]
)

Training shape: (3000888, 6)
First date: 2013-01-01 00:00:00
Last date: 2017-08-15 00:00:00
Unique series: 1782


## 4. Define Forecast Horizon

In [6]:
# Define the final forecast horizon
forecast_start = train_df["date"].max() + pd.Timedelta(days=1)
forecast_dates = pd.date_range(
    start=forecast_start,
    periods=15,
    freq="D"
)

print("Forecast start:", forecast_dates.min())
print("Forecast end:", forecast_dates.max())
print("Forecast days:", len(forecast_dates))

Forecast start: 2017-08-16 00:00:00
Forecast end: 2017-08-30 00:00:00
Forecast days: 15


## 5. Load Future Known Features

The future prediction period contains known calendar and promotion information from the test dataset.

These features are available at forecast time and can therefore be used when preparing the final DeepAR forecast.

In [11]:
# Load test data for future known features
test_df = pd.read_csv(
    project_root / "data" / "raw" / "test.csv"
)

test_df["date"] = pd.to_datetime(test_df["date"])

print("Test shape:", test_df.shape)
print("First date:", test_df["date"].min())
print("Last date:", test_df["date"].max())

Test shape: (28512, 5)
First date: 2017-08-16 00:00:00
Last date: 2017-08-31 00:00:00


In [12]:
# Load the Store × Family series used by the trained DeepAR model
deepar_train = pd.read_parquet(
    processed_path / "train_df.parquet"
)

model_series = deepar_train[
    ["store_nbr", "family"]
].drop_duplicates()

# Keep future known features only for DeepAR-supported series
future_df = test_df[
    test_df["date"].isin(forecast_dates)
].merge(
    model_series,
    on=["store_nbr", "family"],
    how="inner"
)

print("Future data shape:", future_df.shape)
print("First date:", future_df["date"].min())
print("Last date:", future_df["date"].max())

print(
    "Unique series:",
    future_df[["store_nbr", "family"]].drop_duplicates().shape[0]
)

Future data shape: (25920, 5)
First date: 2017-08-16 00:00:00
Last date: 2017-08-30 00:00:00
Unique series: 1728


## 6. Prepare DeepAR Input Data

The final DeepAR forecast uses the same time-series structure as the trained model.

Historical sales are used as the encoder context, while future calendar and promotion features are provided for the forecast horizon.

In [13]:
# Keep only the series used by the trained DeepAR model
history_df = train_df[
    ["date", "store_nbr", "family", "sales", "onpromotion"]
].merge(
    model_series,
    on=["store_nbr", "family"],
    how="inner"
)

# Match the historical start used during DeepAR training
history_df = history_df[
    history_df["date"] >= deepar_train["date"].min()
].copy()

future_input = future_df[
    ["date", "store_nbr", "family", "onpromotion"]
].copy()

future_input["sales"] = np.nan

deepar_input = pd.concat(
    [history_df, future_input],
    ignore_index=True
)

deepar_input = deepar_input.sort_values(
    ["store_nbr", "family", "date"]
).reset_index(drop=True)

print("DeepAR input shape:", deepar_input.shape)
print("First date:", deepar_input["date"].min())
print("Last date:", deepar_input["date"].max())
print(
    "Unique series:",
    deepar_input[["store_nbr", "family"]].drop_duplicates().shape[0]
)

DeepAR input shape: (2885760, 5)
First date: 2013-01-30 00:00:00
Last date: 2017-08-30 00:00:00
Unique series: 1728


## 7. Create DeepAR Time Features

The DeepAR model uses time-based features that are available for both historical and future dates.

These features are derived from the date and do not require future sales values.

In [14]:
# Create calendar features
deepar_input["weekday"] = deepar_input["date"].dt.weekday
deepar_input["month"] = deepar_input["date"].dt.month

# Create a global time index
deepar_input = deepar_input.sort_values(
    ["date", "store_nbr", "family"]
).reset_index(drop=True)

deepar_input["time_idx"] = (
    deepar_input["date"] - deepar_input["date"].min()
).dt.days

print(
    deepar_input[
        ["date", "store_nbr", "family", "sales",
         "onpromotion", "weekday", "month", "time_idx"]
    ].head()
)

print("\nTime index range:",
      deepar_input["time_idx"].min(),
      "→",
      deepar_input["time_idx"].max())

        date  store_nbr        family     sales  onpromotion  weekday  month  \
0 2013-01-30          1    AUTOMOTIVE     6.000            0        2      1   
1 2013-01-30          1        BEAUTY     3.000            0        2      1   
2 2013-01-30          1     BEVERAGES  1031.000            0        2      1   
3 2013-01-30          1         BOOKS     0.000            0        2      1   
4 2013-01-30          1  BREAD/BAKERY   280.501            0        2      1   

   time_idx  
0         0  
1         0  
2         0  
3         0  
4         0  

Time index range: 0 → 1673


## 8. Prepare Final DeepAR Series

The final forecast uses the same Store × Family series structure used during DeepAR training.

Only series supported by the trained model are included in the final prediction dataset.

In [15]:
# Keep only the Store × Family series used by the trained DeepAR model
final_series = deepar_train[
    ["store_nbr", "family"]
].drop_duplicates()

print("Final series:", final_series.shape)
print("Unique series:", final_series.shape[0])

Final series: (1728, 2)
Unique series: 1728


## 9. Create Final DeepAR Dataset

The final DeepAR dataset is prepared using the same series structure and time-based features used during model training.

The dataset includes historical sales for the encoder period and future known features for the 15-day forecast horizon.

In [16]:
# Select the final columns required by DeepAR
final_deepar_data = deepar_input[
    [
        "date",
        "store_nbr",
        "family",
        "sales",
        "onpromotion",
        "weekday",
        "month",
        "time_idx"
    ]
].copy()

print("Final DeepAR dataset shape:", final_deepar_data.shape)
print("Unique series:",
      final_deepar_data[["store_nbr", "family"]].drop_duplicates().shape[0])
print("Missing sales:", final_deepar_data["sales"].isna().sum())

Final DeepAR dataset shape: (2885760, 8)
Unique series: 1728
Missing sales: 25920


## 10. Create Final Prediction Dataset

The final DeepAR dataset is converted into a prediction dataset using the
same time-series structure and feature configuration used during training.

Future sales values are unavailable at forecast time, so placeholder values
are used only to satisfy the dataset structure. The model generates the
actual forecasts for the 15-day prediction horizon.

In [19]:
from pytorch_forecasting import TimeSeriesDataSet
from pytorch_forecasting.data import GroupNormalizer

# Prepare historical data for the dataset definition
final_train_data = final_deepar_data[
    final_deepar_data["sales"].notna()
].copy()

# Match categorical types
for df in [final_train_data, final_deepar_data]:
    df["store_nbr"] = df["store_nbr"].astype(str)
    df["family"] = df["family"].astype(str)
    df["weekday"] = df["weekday"].astype(str)
    df["month"] = df["month"].astype(str)

# Create the same series identifier used during training
final_train_data["series_id"] = (
    final_train_data["store_nbr"]
    + "_"
    + final_train_data["family"]
)

final_deepar_data["series_id"] = (
    final_deepar_data["store_nbr"]
    + "_"
    + final_deepar_data["family"]
)

# Create a prediction copy
prediction_data = final_deepar_data.copy()

# Future sales are unknown.
# Fill only the future target values with a placeholder.
prediction_data["sales"] = prediction_data["sales"].fillna(0)

# Recreate the same TimeSeriesDataSet configuration used during training
training_full = TimeSeriesDataSet(
    final_train_data,
    time_idx="time_idx",
    target="sales",
    group_ids=["series_id"],
    min_encoder_length=15,
    max_encoder_length=30,
    min_prediction_length=1,
    max_prediction_length=15,
    static_categoricals=["store_nbr", "family"],
    time_varying_known_categoricals=["weekday", "month"],
    time_varying_known_reals=["onpromotion"],
    time_varying_unknown_reals=["sales"],
    target_normalizer=GroupNormalizer(
        groups=["series_id"],
        transformation="log1p"
    ),
    add_relative_time_idx=True,
    add_target_scales=True,
    add_encoder_length=True,
    allow_missing_timesteps=True,
)

# Create prediction dataset
prediction_dataset = TimeSeriesDataSet.from_dataset(
    training_full,
    prediction_data,
    predict=True,
    stop_randomization=True,
)

print("Prediction dataset created successfully")
print("Prediction samples:", len(prediction_dataset))
print("Forecast horizon:", 15)
print("Unique series:", prediction_data["series_id"].nunique())

Prediction dataset created successfully
Prediction samples: 1728
Forecast horizon: 15
Unique series: 1728


## 11. Create Prediction DataLoader

The prediction dataset is converted into a DataLoader using the same
configuration used during DeepAR training.

The DataLoader contains all 1,728 Store × Family series for the
15-day forecast horizon.

In [20]:
# Create prediction dataloader
prediction_dataloader = prediction_dataset.to_dataloader(
    train=False,
    batch_size=512,
    num_workers=0
)

print("Prediction DataLoader created successfully")
print("Number of batches:", len(prediction_dataloader))

Prediction DataLoader created successfully
Number of batches: 4


## 12. Load Final DeepAR Model

The trained DeepAR checkpoint is loaded for final forecasting.

No additional training is performed in this notebook.
The loaded model is used only to generate the final 15-day forecast.

In [21]:
from pytorch_forecasting import DeepAR

checkpoint_path = models_path / "deepar_full_trained.ckpt"

final_model = DeepAR.load_from_checkpoint(
    checkpoint_path
)

final_model.eval()

print("Final DeepAR model loaded successfully")

/usr/local/lib/python3.13/dist-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'loss' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['loss'])`.
/usr/local/lib/python3.13/dist-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'logging_metrics' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['logging_metrics'])`.


Final DeepAR model loaded successfully


In [22]:
# Generate final 15-day forecasts
predictions = final_model.predict(
    prediction_dataloader,
    mode="prediction"
)

print("Prediction shape:", predictions.shape)

INFO: GPU available: True (cuda), used: True
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: 💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:lightning.pytorch.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO: 💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
INFO:lightning.pytorch

Prediction shape: torch.Size([1728, 15])


In [23]:
# Convert predictions to numpy
prediction_values = predictions.detach().cpu().numpy()

# Create forecast rows
forecast_output = []

for i, (_, group) in enumerate(
    final_deepar_data.groupby(["store_nbr", "family"], sort=True)
):
    future_dates = group["date"].tail(15).values

    for j, date in enumerate(future_dates):
        forecast_output.append({
            "date": date,
            "store_nbr": group["store_nbr"].iloc[0],
            "family": group["family"].iloc[0],
            "deepar_pred": prediction_values[i, j]
        })

final_forecast = pd.DataFrame(forecast_output)

print("Final forecast shape:", final_forecast.shape)
print("First date:", final_forecast["date"].min())
print("Last date:", final_forecast["date"].max())
print(
    "Unique series:",
    final_forecast[["store_nbr", "family"]].drop_duplicates().shape[0]
)
print("Missing predictions:", final_forecast["deepar_pred"].isna().sum())

Final forecast shape: (25920, 4)
First date: 2017-08-16 00:00:00
Last date: 2017-08-30 00:00:00
Unique series: 1728
Missing predictions: 0


In [25]:
# Clip negative forecasts to zero
negative_count = (final_forecast["deepar_pred"] < 0).sum()

final_forecast["deepar_pred"] = final_forecast["deepar_pred"].clip(lower=0)

print("Negative predictions before clipping:", negative_count)
print(
    "Negative predictions after clipping:",
    (final_forecast["deepar_pred"] < 0).sum()
)
print("Minimum prediction:", final_forecast["deepar_pred"].min())

Negative predictions before clipping: 388
Negative predictions after clipping: 0
Minimum prediction: 0.0


In [26]:
# Final forecast validation checks
print("Rows:", len(final_forecast))
print("Missing predictions:", final_forecast["deepar_pred"].isna().sum())
print("Negative predictions:", (final_forecast["deepar_pred"] < 0).sum())
print(
    "Duplicate rows:",
    final_forecast.duplicated(
        subset=["date", "store_nbr", "family"]
    ).sum()
)
print(
    "Unique dates:",
    final_forecast["date"].nunique()
)
print(
    "Unique series:",
    final_forecast[["store_nbr", "family"]]
    .drop_duplicates()
    .shape[0]
)

Rows: 25920
Missing predictions: 0
Negative predictions: 0
Duplicate rows: 0
Unique dates: 15
Unique series: 1728


In [27]:
# Save final forecast
predictions_path = predictions_path

final_forecast.to_parquet(
    predictions_path / "final_deepar_forecast.parquet",
    index=False
)

final_forecast.to_csv(
    predictions_path / "final_deepar_forecast.csv",
    index=False
)

print("Final forecast saved successfully")
print("Parquet:", predictions_path / "final_deepar_forecast.parquet")
print("CSV:", predictions_path / "final_deepar_forecast.csv")

Final forecast saved successfully
Parquet: /content/drive/MyDrive/DeepAR/DeepAR/predictions/final_deepar_forecast.parquet
CSV: /content/drive/MyDrive/DeepAR/DeepAR/predictions/final_deepar_forecast.csv


### Final Output

The final DeepAR forecast is ready for downstream demand analysis and replenishment recommendations.